In [1]:
import os
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy.engine import URL
from sqlalchemy import create_engine
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

def get_db_engine():
    load_dotenv()
    db_user = os.getenv("DB_USER")
    db_password = os.getenv("DB_PASSWORD")
    db_host = os.getenv("DB_HOST")
    db_port = os.getenv("DB_PORT")
    db_name = os.getenv("DB_NAME")

    if not all([db_user, db_password, db_host, db_port, db_name]):
        raise ValueError("Missing one or more required environment variables.")

    db_url = URL.create(
        drivername="mysql+pymysql",
        username=db_user,
        password=db_password,
        host=db_host,
        port=int(db_port),
        database=db_name
    )
    engine = create_engine(db_url)
    with engine.connect() as conn:
        print("Database connection is made successfully..!\n")
    return engine

def process_catcher_data(engine):
    def clean_catcher_data():
        catcher = pd.read_sql("SELECT jobma_catcher_id, is_premium, subscription_status, company_size, jobma_catcher_parent FROM jobma_catcher WHERE jobma_verified IN (1, '1')", engine).copy()
        catcher.replace("", np.nan, inplace=True)
        catcher['is_premium'] = pd.to_numeric(catcher['is_premium'], errors='coerce').fillna(0).astype(int)
        catcher['subscription_status'] = pd.to_numeric(catcher['subscription_status'], errors='coerce')
        catcher['subscription_status'] = catcher['subscription_status'].replace({2: 0}).fillna(0).astype(int)
        sub_accounts = catcher['jobma_catcher_parent'].value_counts()
        catcher['sub_user'] = catcher['jobma_catcher_id'].map(sub_accounts).fillna(0).astype(int)
        mode_val = catcher['company_size'].mode()[0]
        catcher['company_size'] = catcher['company_size'].fillna(mode_val)
        return catcher

    def clean_wallet_data():
        wallet = pd.read_sql("SELECT catcher_id AS jobma_catcher_id, wallet_amount, is_unlimited FROM wallet", engine).copy()
        wallet['wallet_amount'] = pd.to_numeric(wallet['wallet_amount'], errors='coerce').fillna(0)
        wallet['is_unlimited'] = pd.to_numeric(wallet['is_unlimited'], errors='coerce').fillna(0).astype(int)
        return wallet

    def clean_subscription_data():
        subscription = pd.read_sql("SELECT catcher_id AS jobma_catcher_id, subscription_amount, currency FROM subscription_history", engine).copy()
        subscription['subscription_amount'] = pd.to_numeric(subscription['subscription_amount'], errors='coerce').fillna(0)
        subscription['subscription_amount'] = subscription['subscription_amount'].where(subscription['subscription_amount'] > 0, 0)
        subscription['subscription_amount'] = np.where(subscription['currency'] == 0,subscription['subscription_amount'].round(1),(subscription['subscription_amount'] / 85).round(1))
        subscription = subscription.groupby('jobma_catcher_id').agg(subscription_sum=('subscription_amount', 'sum'),subscription_count=('jobma_catcher_id', 'count')).reset_index()
        return subscription

    def clean_login_data():
        login = pd.read_sql("SELECT jobma_user_id AS jobma_catcher_id, jobma_last_login FROM jobma_login", engine)
        login['jobma_last_login'] = pd.to_datetime(login['jobma_last_login'], errors='coerce')
        today = pd.to_datetime('2024-06-10 11:24:30')  #Need improvements
        login['since_last_login'] = (today - login['jobma_last_login']).dt.days
        login = login.groupby('jobma_catcher_id').agg(since_last_login=('since_last_login', 'min')).reset_index()
        since_login_filled = login['since_last_login'].fillna(float('inf'))
        bins = [0, 30, 90, 180, 365, float('inf')]
        labels = [0, 1, 2, 3, 4]
        login['since_last_login'] = pd.cut(since_login_filled, bins=bins, labels=labels, right=False)
        return login

    def clean_invitations_data():
        invitations = pd.read_sql("SELECT jobma_catcher_id, jobma_interview_mode, jobma_interview_status FROM jobma_pitcher_invitations", engine)
        recorded = invitations[invitations['jobma_interview_mode'].isin([1, '1'])].groupby('jobma_catcher_id').size().reset_index(name='recorded_interview_count')
        live = invitations[invitations['jobma_interview_mode'].isin([2, '2'])].groupby('jobma_catcher_id').size().reset_index(name='live_interview_count')
        invites = invitations[invitations['jobma_interview_status'].isin([0, '0'])].groupby('jobma_catcher_id').size().reset_index(name='invites_count')
        done = invitations[~invitations['jobma_interview_status'].isin([0, '0'])].groupby('jobma_catcher_id').size().reset_index(name='interview_done')
        summary = pd.merge(recorded, live, on='jobma_catcher_id', how='left')
        summary = pd.merge(summary, invites, on='jobma_catcher_id', how='left')
        summary = pd.merge(summary, done, on='jobma_catcher_id', how='left')
        summary.fillna(0, inplace=True)
        return summary

    def clean_pre_recorded_kit_data():
        kit = pd.read_sql("SELECT catcher_id AS jobma_catcher_id FROM job_assessment_kit", engine)
        return kit.groupby('jobma_catcher_id').size().reset_index(name='pre_recorded_kit_counts')

    def clean_job_posting_data():
        postings = pd.read_sql("SELECT jobma_catcher_id FROM jobma_employer_job_posting", engine).copy()
        return postings.groupby('jobma_catcher_id').size().reset_index(name='jobs_posted')

    def working_on_merged_df(work_df):
        columns_to_sum = ['recorded_interview_count', 'live_interview_count', 'invites_count', 'interview_done', 'pre_recorded_kit_counts', 'jobs_posted']
        work_df[columns_to_sum] = work_df[columns_to_sum].fillna(0).astype(int)
        child_sums = work_df[work_df['jobma_catcher_parent'] != 0].groupby('jobma_catcher_parent')[columns_to_sum].sum()
        for col in columns_to_sum:
            work_df.loc[work_df['jobma_catcher_parent'] == 0, col] += work_df.loc[
                work_df['jobma_catcher_parent'] == 0, 'jobma_catcher_id'].map(child_sums[col]).fillna(0)

        sub_ac = work_df[work_df['jobma_catcher_parent'] != 0]
        min_login = sub_ac.groupby('jobma_catcher_parent')['since_last_login'].min().reset_index().rename(columns={'since_last_login': 'min_login'})
        work_df = work_df.merge(min_login, how='left', on='jobma_catcher_parent')
        work_df['since_last_login'] = np.where(work_df['jobma_catcher_parent'] != 0,work_df['min_login'],work_df['since_last_login'])
        work_df.drop(columns=['min_login'], inplace=True)

        work_df = work_df[work_df['jobma_catcher_parent'] == 0].drop('jobma_catcher_parent', axis=1)
        work_df['wallet_amount'] = work_df['wallet_amount'].fillna(0).round(1)
        work_df['subscription_sum'] = work_df['subscription_sum'].fillna(0).round(1)
        work_df['subscription_count'] = work_df['subscription_count'].fillna(0).astype(int)
        work_df['is_unlimited'] = work_df['is_unlimited'].replace('', 1).fillna(0).astype(int)
        work_df['since_last_login'] = work_df['since_last_login'].fillna(4).astype(int)
        return work_df

    catcher_df = clean_catcher_data()
    other_dfs = [clean_wallet_data(),
                 clean_subscription_data(),
                 clean_invitations_data(),
                 clean_pre_recorded_kit_data(),
                 clean_job_posting_data(),
                 clean_login_data()
                ]
    merged_df = catcher_df.copy()
    for df in other_dfs:
        merged_df = pd.merge(merged_df, df, how='left', on='jobma_catcher_id')

    final_df = working_on_merged_df(merged_df)
    return final_df

def get_or_create_processed_data(path='processed_data/processed_data.csv'):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if os.path.exists(path):
        print(f"Loading existing file from : {path}")
        df = pd.read_csv(path)
    else:
        print(f"No existing file found...!!")
        print(f"Creating new data at: {path}")
        engine = get_db_engine()
        df = process_catcher_data(engine)
        df.to_csv(path, index=False)
        print(f"Database is created at {path}...!")
    return df
final_df = get_or_create_processed_data('processed_data/processed_data.csv')
print("df is fetched...!!")
df = final_df.drop('jobma_catcher_id', axis=1)

def get_preprocessor():
    categorical = ['since_last_login']
    ordinal = ['company_size']
    binary = ['is_premium', 'is_unlimited', 'subscription_status']
    numeric = ['sub_user', 'subscription_count', 'live_interview_count', 'interview_done', 'pre_recorded_kit_counts', 'invites_count', 'jobs_posted']
    log_scaled = ['subscription_sum', 'wallet_amount']

    preprocessor = ColumnTransformer([
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                          ('encoder', OneHotEncoder(handle_unknown='ignore'))]), categorical),

        ('ord', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                          ('ordinal', OrdinalEncoder(categories=[["1-25", "26-100", "101-500", "500-1000", "More than 1000"]],
                                                    handle_unknown='use_encoded_value', unknown_value=-1))]), ordinal),

        ('bin', SimpleImputer(strategy='mean'), binary),

        ('num', Pipeline([('imputer', SimpleImputer(strategy='mean')),
                          ('scale', StandardScaler())]), numeric),

        ('log', Pipeline([('imputer', SimpleImputer(strategy='mean')),
                          ('log', FunctionTransformer(np.log1p, validate=False)),
                          ('scale', StandardScaler())]), log_scaled)
    ])
    return preprocessor

def preprocess_data(df, preprocessor=None, fit=True):
    if fit or preprocessor is None:
        preprocessor = get_preprocessor()
        features = preprocessor.fit_transform(df)
    else:
        features = preprocessor.transform(df)
    return features, preprocessor

# --- Autoencoder ---
class ClientAutoencoder(nn.Module):
    def __init__(self, input_dim, embedding_dim=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, embedding_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(embedding_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, input_dim)
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return encoded, decoded

def to_tensor(X):
    return torch.FloatTensor(X.toarray() if hasattr(X, 'toarray') else X)

# --- Training with Early Stopping ---
def train_autoencoder(X_all, model_path, input_dim, epochs=100, batch_size=32, patience=10):
    X_train, X_val = train_test_split(X_all, test_size=0.2, random_state=42)
    train_tensor = to_tensor(X_train)
    val_tensor = to_tensor(X_val)

    train_loader = DataLoader(TensorDataset(train_tensor, train_tensor), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(val_tensor, val_tensor), batch_size=batch_size)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = ClientAutoencoder(input_dim).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

    best_loss = float('inf')
    patience_counter = 0
    best_state = None

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for batch in train_loader:
            inputs, _ = batch
            inputs = inputs.to(device)

            optimizer.zero_grad()
            _, outputs = model(inputs)
            loss = criterion(outputs, inputs)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                inputs, _ = batch
                inputs = inputs.to(device)
                _, outputs = model(inputs)
                val_loss += criterion(outputs, inputs).item()
        val_loss /= len(val_loader)

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        if val_loss < best_loss:
            best_loss = val_loss
            best_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping triggered.")
                break

    if best_state:
        model.load_state_dict(best_state)
        torch.save(model.state_dict(), model_path)

    return model

# --- Model Management ---
def load_or_train_autoencoder(X, model_path='client_autoencoder1.pth'):
    input_dim = X.shape[1]
    model = ClientAutoencoder(input_dim)
    if os.path.exists(model_path):
        model.load_state_dict(torch.load(model_path, map_location='cpu'))
        print("Model loaded from disk.")
    else:
        print("No model found. Training new model...")
        model = train_autoencoder(X, model_path, input_dim)
    return model

# --- Embedding Extraction ---
def extract_embeddings(model, X):
    model.eval()
    with torch.no_grad():
        tensor = to_tensor(X)
        embeddings, _ = model(tensor)
    return embeddings.cpu().numpy()

# --- Cosine Similarity Recommendation ---
def generate_recommendations(embeddings, client_ids, top_k=5, min_similarity=0.5):
    similarity_matrix = cosine_similarity(embeddings)
    recommendations = {}
    for idx, client_id in enumerate(client_ids):
        sim_scores = similarity_matrix[idx]
        top_indices = np.argsort(sim_scores)[::-1]
        similar_clients = []
        for i in top_indices:
            if client_ids[i] == client_id:
                continue
            if sim_scores[i] >= min_similarity:
                similar_clients.append((client_ids[i], round(sim_scores[i], 4)))
            if len(similar_clients) >= top_k:
                break
        recommendations[client_id] = similar_clients
    return recommendations

# --- Entry Point ---
def get_similar_clients(input_client_id, df, top_k=5, min_similarity=0.2):
    client_ids = df['jobma_catcher_id'].values
    df_cleaned = df.drop('jobma_catcher_id', axis=1)

    features, preprocessor = preprocess_data(df_cleaned, fit=True)
    model = load_or_train_autoencoder(features)
    embeddings = extract_embeddings(model, features)
    recs = generate_recommendations(embeddings, client_ids, top_k, min_similarity)

    if input_client_id not in recs:
        print(f"Client ID {input_client_id} not in dataset.")
        return pd.DataFrame()

    similar_ids = [cid for cid, _ in recs[input_client_id]]
    scores = [score for _, score in recs[input_client_id]]
    result = df[df['jobma_catcher_id'].isin(similar_ids)].copy()
    result['similarity_score'] = scores
    return result.reset_index(drop=True)

# --- Run ---
if __name__ == "__main__":
    engine = get_db_engine()
    final_df = process_catcher_data(engine)

    df = final_df
    input_id = 10521
    recommendations_df = get_similar_clients(input_id, df)
recommendations_df

Loading existing file from : processed_data/processed_data.csv
df is fetched...!!
Database connection is made successfully..!

No model found. Training new model...
Epoch 1/100 | Train Loss: 0.5529 | Val Loss: 0.1883
Epoch 2/100 | Train Loss: 0.3305 | Val Loss: 0.1175
Epoch 3/100 | Train Loss: 0.2919 | Val Loss: 0.0753
Epoch 4/100 | Train Loss: 0.2641 | Val Loss: 0.0813
Epoch 5/100 | Train Loss: 0.2597 | Val Loss: 0.0505
Epoch 6/100 | Train Loss: 0.2208 | Val Loss: 0.0782
Epoch 7/100 | Train Loss: 0.2063 | Val Loss: 0.0620
Epoch 8/100 | Train Loss: 0.1949 | Val Loss: 0.0612
Epoch 9/100 | Train Loss: 0.2139 | Val Loss: 0.0486
Epoch 10/100 | Train Loss: 0.1630 | Val Loss: 0.0384
Epoch 11/100 | Train Loss: 0.1130 | Val Loss: 0.0349
Epoch 12/100 | Train Loss: 0.2595 | Val Loss: 0.0339
Epoch 13/100 | Train Loss: 0.1944 | Val Loss: 0.0355
Epoch 14/100 | Train Loss: 0.2145 | Val Loss: 0.0292
Epoch 15/100 | Train Loss: 0.2169 | Val Loss: 0.0406
Epoch 16/100 | Train Loss: 0.1781 | Val Loss: 0.0

,jobma_catcher_id,is_premium,subscription_status,company_size,sub_user,wallet_amount,is_unlimited,subscription_sum,subscription_count,recorded_interview_count,live_interview_count,invites_count,interview_done,pre_recorded_kit_counts,jobs_posted,since_last_login,similarity_score
0,9620,0,1,1-25,0,500000.0,1,1.2,1,9,0,0,9,5,8,2,0.9135
1,9891,0,1,26-100,1,500000.0,1,117.6,1,65,16,0,81,19,18,2,0.8955
2,10172,0,1,26-100,2,500000.0,1,123.6,6,45,0,0,45,12,6,1,0.8906
3,10240,0,1,1-25,1,500000.0,1,117.6,1,7,2,0,9,16,9,2,0.8797
4,10489,0,1,101-500,1,500000.0,1,118.8,2,35,0,2,33,4,5,0,0.8601


In [2]:
input_client_id = 10240
similar_clients_df = get_similar_clients(input_client_id, df)
similar_clients_df

Model loaded from disk.


,jobma_catcher_id,is_premium,subscription_status,company_size,sub_user,wallet_amount,is_unlimited,subscription_sum,subscription_count,recorded_interview_count,live_interview_count,invites_count,interview_done,pre_recorded_kit_counts,jobs_posted,since_last_login,similarity_score
0,2656,0,1,1-25,1,66666.0,1,176.5,1,7,0,0,7,4,2,1,0.9360
1,9891,0,1,26-100,1,500000.0,1,117.6,1,65,16,0,81,19,18,2,0.9345
2,9944,0,1,1-25,0,500000.0,1,117.6,1,8,1,0,9,2,6,3,0.9309
3,10172,0,1,26-100,2,500000.0,1,123.6,6,45,0,0,45,12,6,1,0.9231
4,10444,0,1,1-25,1,500000.0,1,120.6,3,11,3,0,14,3,2,0,0.9183


In [3]:
input_client_id = 10259
similar_clients_df = get_similar_clients(input_client_id, df)
similar_clients_df

Model loaded from disk.


,jobma_catcher_id,is_premium,subscription_status,company_size,sub_user,wallet_amount,is_unlimited,subscription_sum,subscription_count,recorded_interview_count,live_interview_count,invites_count,interview_done,pre_recorded_kit_counts,jobs_posted,since_last_login,similarity_score
0,9780,1,1,101-500,3,10445.0,0,0.6,1,53,1,0,54,4,19,2,0.9360
1,9931,0,1,26-100,1,296.0,0,0.6,1,32,0,0,32,6,6,2,0.9198
2,9969,0,1,1-25,1,188.0,0,0.6,1,33,0,0,33,6,9,3,0.9061
3,10155,0,1,26-100,1,195.0,0,0.6,1,20,0,0,20,4,10,2,0.8989
4,10162,1,1,1-25,2,4074.0,0,0.6,1,27,0,0,27,8,2,2,0.8871


In [4]:
input_client_id = 10095
similar_clients_df = get_similar_clients(input_client_id, df)
similar_clients_df

Model loaded from disk.


,jobma_catcher_id,is_premium,subscription_status,company_size,sub_user,wallet_amount,is_unlimited,subscription_sum,subscription_count,recorded_interview_count,live_interview_count,invites_count,interview_done,pre_recorded_kit_counts,jobs_posted,since_last_login,similarity_score
0,9602,0,1,1-25,1,99.0,0,0.6,1,26,0,0,26,20,17,3,0.9344
1,9871,0,1,1-25,0,47.0,0,0.6,1,4,0,0,4,3,12,3,0.9080
2,10059,0,1,1-25,0,33.0,0,0.6,1,4,0,1,3,3,3,1,0.9005
3,10216,0,1,1-25,0,50.0,0,0.0,1,0,0,0,0,1,5,2,0.8933
4,10397,0,1,1-25,1,772.0,0,1.2,1,146,1,0,147,20,23,1,0.8909


In [5]:
input_client_id = 6645
similar_clients_df = get_similar_clients(input_client_id, df)
similar_clients_df

Model loaded from disk.


,jobma_catcher_id,is_premium,subscription_status,company_size,sub_user,wallet_amount,is_unlimited,subscription_sum,subscription_count,recorded_interview_count,live_interview_count,invites_count,interview_done,pre_recorded_kit_counts,jobs_posted,since_last_login,similarity_score
0,3941,1,0,500-1000,0,60.0,0,176.5,1,0,0,0,0,0,0,4,1.0000
1,3992,1,0,500-1000,0,60.0,0,208.2,1,0,0,0,0,0,0,4,1.0000
2,5546,1,0,500-1000,0,60.0,0,208.2,1,0,0,0,0,0,0,4,1.0000
3,7367,1,0,500-1000,0,60.0,0,176.5,1,0,0,0,0,0,0,4,0.9997
4,8240,1,0,500-1000,0,60.0,0,176.5,1,0,0,0,0,0,0,4,0.9997


In [6]:
input_client_id = 6554
similar_clients_df = get_similar_clients(input_client_id, df)
similar_clients_df

Model loaded from disk.


,jobma_catcher_id,is_premium,subscription_status,company_size,sub_user,wallet_amount,is_unlimited,subscription_sum,subscription_count,recorded_interview_count,live_interview_count,invites_count,interview_done,pre_recorded_kit_counts,jobs_posted,since_last_login,similarity_score
0,2997,0,0,26-100,0,60.0,0,0.6,1,0,0,0,0,0,0,4,1.0
1,3078,0,0,26-100,0,60.0,0,0.6,1,0,0,0,0,0,0,4,1.0
2,3241,0,0,26-100,0,60.0,0,0.6,1,0,0,0,0,0,0,4,1.0
3,3309,0,0,26-100,0,60.0,0,0.6,1,0,0,0,0,0,0,4,1.0
4,3636,0,0,26-100,0,60.0,0,0.6,1,0,0,0,0,0,0,4,1.0


In [7]:
final_df[final_df["jobma_catcher_id"] == 2935]

,jobma_catcher_id,is_premium,subscription_status,company_size,sub_user,wallet_amount,is_unlimited,subscription_sum,subscription_count,recorded_interview_count,live_interview_count,invites_count,interview_done,pre_recorded_kit_counts,jobs_posted,since_last_login
1,2935,0,0,26-100,0,60.0,0,0.6,1,0,0,0,0,0,0,2


In [8]:
final_df[final_df["jobma_catcher_id"] == 2997]

,jobma_catcher_id,is_premium,subscription_status,company_size,sub_user,wallet_amount,is_unlimited,subscription_sum,subscription_count,recorded_interview_count,live_interview_count,invites_count,interview_done,pre_recorded_kit_counts,jobs_posted,since_last_login
60,2997,0,0,26-100,0,60.0,0,0.6,1,0,0,0,0,0,0,4
